Summary of Findings — Student Performance Regression

This project examines whether behavioral and demographic factors (excluding prior grades) can predict students' final math grades, using a multiple linear regression on 397 students.

Model fit: R² = 0.186 (F-statistic p < 0.001) — the model is statistically significant overall, though behavioral/demographic factors alone explain a modest share of grade variance, as expected once prior performance (G1, G2) is excluded to avoid data leakage.
Strongest predictor: Number of past class failures (coef = -1.83, p < 0.001) — each additional failure is associated with a ~1.83 point drop in final grade, by far the largest effect in the model.
Other significant factors: Mother's education level (coef = +0.58, p = 0.024) and frequency of going out with friends (coef = -0.61, p = 0.006) were also statistically significant predictors.
No multicollinearity: VIF values for all predictors were below 2, confirming the model's coefficients are reliable and not distorted by correlated predictors.
Not significant: Study time, absences, health, father's education, and travel time did not show statistically significant effects in this model — a legitimate finding, not a null result to hide.

In [5]:
import pandas as pd

df_math = pd.read_excel(r"C:\Users\NANDINI\OneDrive\Desktop\project\student performance\Maths.csv")

print(df_math.shape)
print(df_math.head())
print(df_math.columns.tolist())

(397, 33)
  school sex  age address famsize Pstatus  Medu  Fedu     Mjob      Fjob  ...  \
0     GP   F   18       U     GT3       A     4     4  at_home   teacher  ...   
1     GP   F   17       U     GT3       T     1     1  at_home     other  ...   
2     GP   F   15       U     LE3       T     1     1  at_home     other  ...   
3     GP   F   15       U     GT3       T     4     2   health  services  ...   
4     GP   F   16       U     GT3       T     3     3    other     other  ...   

  famrel freetime  goout  Dalc  Walc health absences  G1  G2  G3  
0      4        3      4     1     1      3        6   5   6   6  
1      5        3      3     1     1      3        4   5   5   6  
2      4        3      2     2     3      3       10   7   8  10  
3      3        2      2     1     1      5        2  15  14  15  
4      4        3      2     1     2      5        4   6  10  10  

[5 rows x 33 columns]
['school', 'sex', 'age', 'address', 'famsize', 'Pstatus', 'Medu', 'Fedu', 'Mjo

In [6]:
import matplotlib.pyplot as plt
import seaborn as sns

# Correlation of numeric features with G3
numeric_df = df_math.select_dtypes(include='number')
correlations = numeric_df.corr()['G3'].sort_values(ascending=False)
print(correlations)

G3            1.000000
G2            0.905238
G1            0.802676
Medu          0.220783
Fedu          0.155256
studytime     0.104015
famrel        0.050468
absences      0.037944
freetime      0.013131
Dalc         -0.049826
Walc         -0.052185
health       -0.065737
traveltime   -0.122475
goout        -0.127760
age          -0.172175
failures     -0.361237
Name: G3, dtype: float64


In [7]:
import statsmodels.api as sm

# Select predictors, excluding G1, G2, G3
predictors = ['Medu', 'Fedu', 'studytime', 'failures', 'famrel', 
              'absences', 'freetime', 'goout', 'Dalc', 'Walc', 
              'health', 'traveltime', 'age']

X = df_math[predictors]
y = df_math['G3']

X = sm.add_constant(X)  # adds intercept term

model = sm.OLS(y, X).fit()
print(model.summary())

                            OLS Regression Results                            
Dep. Variable:                     G3   R-squared:                       0.186
Model:                            OLS   Adj. R-squared:                  0.159
Method:                 Least Squares   F-statistic:                     6.753
Date:                Mon, 24 Aug 2026   Prob (F-statistic):           1.03e-11
Time:                        15:11:10   Log-Likelihood:                -1128.1
No. Observations:                 397   AIC:                             2284.
Df Residuals:                     383   BIC:                             2340.
Df Model:                          13                                         
Covariance Type:            nonrobust                                         
                 coef    std err          t      P>|t|      [0.025      0.975]
------------------------------------------------------------------------------
const         14.1205      3.333      4.236      0.0

In [8]:
from statsmodels.stats.outliers_influence import variance_inflation_factor

vif_data = pd.DataFrame()
vif_data["feature"] = X.columns
vif_data["VIF"] = [variance_inflation_factor(X.values, i) for i in range(X.shape[1])]
print(vif_data)

       feature         VIF
0        const  247.293991
1         Medu    1.767906
2         Fedu    1.717566
3    studytime    1.133144
4     failures    1.198668
5       famrel    1.075307
6     absences    1.082496
7     freetime    1.182470
8        goout    1.349267
9         Dalc    1.803636
10        Walc    2.083719
11      health    1.048654
12  traveltime    1.067888
13         age    1.153739
